In [1]:
%pip install python-dotenv
from dotenv import load_dotenv
load_dotenv("../.env")


[notice] A new release of pip is available: 24.1 -> 24.3.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


True

In [2]:
import os
from lightrag import LightRAG, QueryParam
from lightrag.llm import gpt_4o_mini_complete, gpt_4o_complete


#########
# Uncomment the below two lines if running in a jupyter notebook to handle the async nature of rag.insert()
import nest_asyncio
nest_asyncio.apply()
#########

WORKING_DIR = "../lightrag-dev/dna"


if not os.path.exists(WORKING_DIR):
    os.mkdir(WORKING_DIR)

rag = LightRAG(
    working_dir=WORKING_DIR,
    llm_model_func=gpt_4o_mini_complete  # Use gpt_4o_mini_complete LLM model
    # llm_model_func=gpt_4o_complete  # Optionally, use a stronger model
)

with open("../lightrag-dev/dna.md") as f:
    rag.insert(f.read())


/Users/boshi/Library/Caches/pypoetry/virtualenvs/lightrag-h2Bk8IZY-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:lightrag:Logger initialized for working directory: ../lightrag-dev/dna
DEBUG:lightrag:LightRAG init with param:
  working_dir = ../lightrag-dev/dna,
  chunk_token_size = 1200,
  chunk_overlap_token_size = 100,
  tiktoken_model_name = gpt-4o-mini,
  entity_extract_max_gleaning = 1,
  entity_summary_to_max_tokens = 500,
  node_embedding_algorithm = node2vec,
  node2vec_params = {'dimensions': 1536, 'num_walks': 10, 'walk_length': 40, 'window_size': 2, 'iterations': 3, 'random_seed': 3},
  embedding_func = {'embedding_dim': 1536, 'max_token_size': 8192, 'func': <function openai_embedding at 0x3081ef380>},
  embedding_batch_num = 32,
  embedding_func_max_async = 16,
  llm_model_func

In [3]:
import logging
logging.getLogger("openai").setLevel(logging.ERROR)
logging.getLogger("httpx").setLevel(logging.ERROR)
print("local search\n")
# Perform hybrid search
print(rag.query("What is the test result and findings? reply in chinese", param=QueryParam(mode="local")))
print("--------------------------------")

print("global search\n")
# Perform hybrid search
print(rag.query("What is the test result and findings? reply in chinese", param=QueryParam(mode="global")))
print("--------------------------------")

print("hybrid search\n")
# Perform hybrid search
print(rag.query("What is the test result and findings? reply in chinese", param=QueryParam(mode="hybrid")))
print("--------------------------------")

print("naive search\n")
# Perform hybrid search
print(rag.query("What is the test result and findings? reply in chinese", param=QueryParam(mode="naive")))
print("--------------------------------")


local search



INFO:lightrag:Local query uses 60 entites, 130 relations, 27 text units


{'relations': [{'id': 0, 'source': '"HEALTHCARE PROVIDER"', 'target': '"PROTEOMIC TEST"', 'description': '"Healthcare Providers help patients interpret the results of the Proteomic Test to manage cardiovascular risks effectively."', 'keywords': '"patient care, risk management"', 'weight': 7.0, 'rank': 43}, {'id': 1, 'source': '"PATIENT"', 'target': '"PROTEOMIC TEST"', 'description': '"The Patient is the subject of the Proteomic Test, though their symptoms and family history are not part of the analysis."', 'keywords': '"medical research, patient analysis"', 'weight': 7.0, 'rank': 40}, {'id': 2, 'source': '"GLUCOSE TOLERANCE"', 'target': '"PROTEOMIC TEST"', 'description': '"Glucose Tolerance is a critical measurement included in the Proteomic Test to evaluate metabolic health."', 'keywords': '"metabolic health, clinical evaluation"', 'weight': 9.0, 'rank': 37}, {'id': 3, 'source': '"DEMENTIA RISK"', 'target': '"PROTEOMIC TEST"', 'description': '"The Proteomic Test is designed to analyze

INFO:lightrag:Global query uses 70 entites, 60 relations, 25 text units


{'relations': [{'id': 0, 'source': '"PATIENT"', 'target': '"PROTEOMIC TEST"', 'description': '"The Patient is the subject of the Proteomic Test, though their symptoms and family history are not part of the analysis."', 'keywords': '"medical research, patient analysis"', 'weight': 7.0, 'rank': 40}, {'id': 1, 'source': '"PROTEOMIC TEST"', 'target': '"SYMPTOMS"', 'description': '"The Proteomic Test examines proteins independently of Symptoms that may indicate conditions like Dementia."', 'keywords': '"medical analysis, symptom exclusion"', 'weight': 6.0, 'rank': 34}, {'id': 2, 'source': '"LUNG CANCER RISK"', 'target': '"PROTEOMIC TEST"', 'description': '"The Proteomic Test encompasses an evaluation for Lung Cancer Risk through protein analysis."."', 'keywords': '"health assessment, predictive analysis"', 'weight': 8.0, 'rank': 33}, {'id': 3, 'source': '"PROTEOMIC TEST"', 'target': '"YOUR HEALTH"', 'description': '"Your Health employs the Proteomic Test as part of its analysis strategy for

INFO:lightrag:Local query uses 60 entites, 130 relations, 27 text units
INFO:lightrag:Global query uses 70 entites, 60 relations, 25 text units


{'relations': [{'id': 0, 'source': '"HEALTHCARE PROVIDER"', 'target': '"PROTEOMIC TEST"', 'description': '"Healthcare Providers help patients interpret the results of the Proteomic Test to manage cardiovascular risks effectively."', 'keywords': '"patient care, risk management"', 'weight': 7.0, 'rank': 43}, {'id': 1, 'source': '"PATIENT"', 'target': '"PROTEOMIC TEST"', 'description': '"The Patient is the subject of the Proteomic Test, though their symptoms and family history are not part of the analysis."', 'keywords': '"medical research, patient analysis"', 'weight': 7.0, 'rank': 40}, {'id': 2, 'source': '"GLUCOSE TOLERANCE"', 'target': '"PROTEOMIC TEST"', 'description': '"Glucose Tolerance is a critical measurement included in the Proteomic Test to evaluate metabolic health."', 'keywords': '"metabolic health, clinical evaluation"', 'weight': 9.0, 'rank': 37}, {'id': 3, 'source': '"DEMENTIA RISK"', 'target': '"PROTEOMIC TEST"', 'description': '"The Proteomic Test is designed to analyze

In [4]:
import networkx as nx
from pyvis.network import Network
import random

# Load the GraphML file
G = nx.read_graphml("../lightrag-dev/dna/graph_chunk_entity_relation.graphml")

# Create a Pyvis network
net = Network(height="100vh", notebook=True)

# Convert NetworkX graph to Pyvis network
net.from_nx(G)

# Add colors to nodes
for node in net.nodes:
    node["color"] = "#{:06x}".format(random.randint(0, 0xFFFFFF))

# Save and display the network
net.show("knowledge_graph.html")


knowledge_graph.html
